In [15]:
import pandas as pd
from data_preprocessing import create_train_test_val_sets, get_processed_df
import joblib
import os
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.feature_selection import SelectKBest, VarianceThreshold, chi2, f_classif, mutual_info_classif
from sklearn.base import clone
from collections import Counter
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [7]:
#Create test train splits
x_mendeley, y_mendeley = get_processed_df(r"..\data\raw\Mendeley Dataset.csv")
x_kaggle, y_kaggle= get_processed_df(r"..\data\raw\dataset_phishing.csv")

mendeley_sets = create_train_test_val_sets(x_mendeley,y_mendeley, label_col="Label", test_size=0.2, n_splits=5)
kaggle_sets = create_train_test_val_sets(x_kaggle,y_kaggle, label_col="Label", test_size=0.2, n_splits=5)

----------Processing None Dataset----------

Class Distribution:
Label
0    0.518415
1    0.481585
Name: proportion, dtype: float64
int64

Total Missing Values: 0
No categorical features to hash
Shape After Processing: (247950, 42)
True
----------Processing None Dataset----------

Class Distribution:
Label
legitimate    0.5
phishing      0.5
Name: proportion, dtype: float64
object

Total Missing Values: 0
Shape After Processing: (11430, 32856)
True
Train/validation/test split prepared: 210757 instances for training, 37193 instances for validation, 49590 instances for testing
Stratified 5-fold CV splits created.
Train/validation/test split prepared: 9715 instances for training, 1715 instances for validation, 2286 instances for testing
Stratified 5-fold CV splits created.


In [8]:
kaggle_sets["y_train"] = kaggle_sets["y_train"].astype(int)
kaggle_sets["y_val"] = kaggle_sets["y_val"].astype(int)
kaggle_sets["y_test"] = kaggle_sets["y_test"].astype(int)

In [30]:
#import phase 1 models
xgb_mendeley = joblib.load('./models/phase_1/xgboost_mendeley_no_fs.joblib')
xgb_kaggle = joblib.load('./models/phase_1/xgboost_kaggle_no_fs.joblib')
logreg_mendeley = joblib.load('./models/phase_1/logreg_mendeley_no_fs.joblib')
logreg_kaggle = joblib.load('./models/phase_1/logreg_kaggle_no_fs.joblib')
# rf_mendeley = joblib.load('./models/phase_1/rf_mendeley_no_fs.joblib')
# rf_kaggle = joblib.load('./models/phase_1/rf_kaggle_no_fs.joblib')

In [ ]:
def run_stability_algorithm(ds_name, ds_set, model,  fs_params,fs_method='anova', stability_threshold=0.6):
    """
    Uses pre-calculated cv_splits to determine feature stability.
    """
    x_train_full = ds_set["x_train"]
    y_train_full = ds_set["y_train"]
    splits = ds_set["cv_splits"]
    
    fold_selected_features = []
    
    # Define FS Strategies
    selectors = {
        'chi2': chi2,
        'anova': f_classif,
        'mi': mutual_info_classif,
        'variance': lambda v: VarianceThreshold(threshold=v)
    }

    print(f"--- Stability Search: {ds_name} | FS: {fs_method} ---")

    #Iterate through pre-defined splits
    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        # Create fold-specific data using iloc for index-based slicing
        x_fold_train = x_train_full.iloc[train_idx]
        y_fold_train = y_train_full.iloc[train_idx]

        #Apply the chosen FS method
        if fs_method == 'variance':
            selector = VarianceThreshold(threshold=fs_params)
            selector.fit(x_fold_train)
        else:
            # Ensure non-negative for chi2
            score_func = selectors[fs_method]
            selector = SelectKBest(score_func=score_func, k=fs_params)
            selector.fit(x_fold_train, y_fold_train)

        # Record winning features for this fold
        winning_feats = x_train_full.columns[selector.get_support()].tolist()
        fold_selected_features.append(winning_feats)

    #Compute Stability Metrics
    all_fold_features = [f for sublist in fold_selected_features for f in sublist] #flatten list of lists
    counts = Counter(all_fold_features) #dict of feature: count across folds
    num_folds = len(splits)
    
    # Filter by threshold
    stable_features = [f for f, count in counts.items() if (count / num_folds) >= stability_threshold]

    #Final Training on the Stable subset
    model.fit(x_train_full[stable_features], y_train_full)
    
    # Evaluate on Validation Set
    y_val_pred = model.predict(ds_set["x_val"][stable_features])
    val_f1 = f1_score(ds_set["y_val"], y_val_pred)

    return {
        "method": fs_method,
        "stable_features": stable_features,
        "val_f1": val_f1,
        "feature_freq": dict(counts)
    }

In [ ]:
stability_report = []
#['anova', 'chi2', 'mi', 'variance']

fs_methods_and_params_mendeley={
    'anova': 30,
    'chi2': 25,
    'mi': 25,
    'variance': 0.05
}
fs_methods_and_k_values_kaggle={
    'anova': 10000
    # 'chi2': 20,
    # 'mi': 20,
    # 'variance': 20
}

# Run for Mendeley
for fs, k in fs_methods_and_params_mendeley.items():
    # Use your function to get the stable features for this method
    result_xgb = run_stability_algorithm("Mendeley", mendeley_sets, xgb_mendeley, k, fs_method=fs, )
    results_logreg = run_stability_algorithm("Mendeley", mendeley_sets, logreg_mendeley, k, fs_method=fs)
    # rf_results = run_stability_algorithm("Mendeley", mendeley_sets, rf_mendeley, k, fs_method=fs)
    stability_report.append({
        'FS_Method': fs,
        'Stability_Score': stability_score,
        'Avg_Val_F1': result_xgb['val_f1'],
        'Stable_Features': result_xgb['stable_features']
    })

# Convert to DF to see the results more clearly
report_df = pd.DataFrame(stability_report)

#save stability report
report_df.to_csv('./stability_report_mendeley.csv', index=False)
print(report_df.sort_values(by='Stability_Score', ascending=False))

--- Stability Search: Mendeley | FS: anova ---


c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
c:\Python312\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
